# Extracellular diffusion Monte Carlo calibration of WM tortuosity exponents

This standalone notebook tests the proposed WM extracellular tortuosity law **without using that law to generate the diffusion data**. It simulates extracellular Brownian walkers in periodic 3-D substrates containing impermeable spherical soma and parallel cylindrical neurites (aligned with z), with no CSF/free-water compartment.

Effective diffusivities are measured from displacement:

$$D_{\parallel}(t)=\langle\Delta z^2\rangle/(2t), \qquad D_{\perp}(t)=\langle\Delta x^2+\Delta y^2\rangle/(4t).$$

Only afterwards are the candidate laws tested:

$$D_{\parallel}\approx D_0\left(1-\frac{f_s}{1-f_n}\right)^a,$$

$$D_{\perp}\approx D_{\parallel}\left(1-\frac{f_n}{1-f_s}\right)^b.$$

Initial experiment: $D_0=2.2$ um2/ms, diffusion time 12 ms, neurite radius 0.5 um, $f_s=0.05:0.05:0.30$, $f_n=0.20:0.10:0.50$, and soma radius 3:1:8 um.

**Collision update:** blocked Brownian endpoints are no longer discarded. Each proposed segment is propagated to its first sphere/cylinder intersection and the remaining displacement is reflected about the local surface normal. Multiple collisions per timestep are allowed up to a safeguard limit.

**Important numerical note:** specular reflection of a finite Brownian proposal is still a finite-step approximation to reflecting Brownian motion. Timestep convergence remains mandatory.

**Separate geometry limitation:** the current random-sequential substrate packer can fail at the middle/high mixed fractions. Keep production disabled until the packing engine has also been validated or replaced.

In [1]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_SEED = 12345
D0 = 2.2
T_MAX = 12.0
R_NEURITE = 0.5
F_SOMA_VALUES = np.arange(0.05, 0.30 + 1e-9, 0.05)
F_NEURITE_VALUES = np.arange(0.20, 0.50 + 1e-9, 0.10)
R_SOMA_VALUES = np.arange(3.0, 8.0 + 1e-9, 1.0)
SAVE_TIMES = np.array([1.0, 2.0, 4.0, 8.0, 12.0])
OUTDIR = Path("mc_tortuosity_outputs")
OUTDIR.mkdir(exist_ok=True)
print(F_SOMA_VALUES, F_NEURITE_VALUES, R_SOMA_VALUES)

[0.05 0.1  0.15 0.2  0.25 0.3 ] [0.2 0.3 0.4 0.5] [3. 4. 5. 6. 7. 8.]


## Geometry and collision helpers

Cylinders are infinite along z and periodic in x/y. Spheres are periodic in all three dimensions. Realised fractions are recorded because integer object counts discretise the requested fractions.

In [1]:
# =============================================================================
# SUBSTRATE AND BASIC GEOMETRY HELPERS
# =============================================================================

from dataclasses import dataclass
import numpy as np


@dataclass
class Substrate:

    box_length: float

    r_neurite: float

    cylinder_xy: np.ndarray

    sphere_xyz: np.ndarray

    sphere_r: float

    substrate_type: str

    # Geometry diagnostics
    f_soma_real: float = np.nan

    f_neurite_real: float = np.nan

    f_neurite_raw: float = np.nan

    soma_neurite_overlap: float = np.nan

    geometry_kind: str = "periodic_lattice_union"


def periodic_delta(delta, L):

    return (
        delta
        - L * np.round(
            delta / L
        )
    )


def inside_cylinders(
    points,
    centers,
    radius,
    L,
    chunk=256,
):

    if len(centers) == 0:

        return np.zeros(
            len(points),
            dtype=bool,
        )

    out = np.zeros(
        len(points),
        dtype=bool,
    )

    r2 = radius**2

    for j in range(
        0,
        len(centers),
        chunk,
    ):

        c = centers[
            j:j + chunk
        ]

        dx = periodic_delta(
            points[:, None, 0]
            - c[None, :, 0],
            L,
        )

        dy = periodic_delta(
            points[:, None, 1]
            - c[None, :, 1],
            L,
        )

        out |= np.any(
            dx*dx + dy*dy < r2,
            axis=1,
        )

    return out


def inside_spheres(
    points,
    centers,
    radius,
    L,
    chunk=128,
):

    if len(centers) == 0:

        return np.zeros(
            len(points),
            dtype=bool,
        )

    out = np.zeros(
        len(points),
        dtype=bool,
    )

    r2 = radius**2

    for j in range(
        0,
        len(centers),
        chunk,
    ):

        c = centers[
            j:j + chunk
        ]

        d = periodic_delta(
            points[:, None, :]
            - c[None, :, :],
            L,
        )

        out |= np.any(
            np.sum(
                d*d,
                axis=2,
            )
            < r2,
            axis=1,
        )

    return out


def inside_obstacles(
    points,
    sub,
):

    return (

        inside_cylinders(
            points,
            sub.cylinder_xy,
            sub.r_neurite,
            sub.box_length,
        )

        |

        inside_spheres(
            points,
            sub.sphere_xyz,
            sub.sphere_r,
            sub.box_length,
        )

    )

In [3]:
def make_substrate(f_soma, f_neurite, r_soma, *, box_length=40.0, r_neurite=0.5, seed=0, max_attempts=20000, substrate_type="mixed"):
    rng=np.random.default_rng(seed); L=float(box_length)
    n_cyl=int(round(f_neurite*L**2/(np.pi*r_neurite**2)))
    n_sph=0 if f_soma==0 else int(round(f_soma*L**3/((4/3)*np.pi*r_soma**3)))
    cylinders=[]; spheres=[]
    for i in range(n_sph):
        for attempt in range(max_attempts):
            c=rng.uniform(0,L,3)
            if sph_clear_sph(c,np.asarray(spheres),r_soma,L) and sph_clear_cyl(c,np.asarray(cylinders),r_soma,r_neurite,L):
                spheres.append(c); break
        else: raise RuntimeError(f"Sphere packing failed {i+1}/{n_sph}")
    for i in range(n_cyl):
        for attempt in range(max_attempts):
            c=rng.uniform(0,L,2)
            if cyl_clear_cyl(c,np.asarray(cylinders),r_neurite,L) and cyl_clear_sph(c,np.asarray(spheres),r_neurite,r_soma,L):
                cylinders.append(c); break
        else: raise RuntimeError(f"Cylinder packing failed {i+1}/{n_cyl}")
    cylinders=np.asarray(cylinders,float).reshape(-1,2); spheres=np.asarray(spheres,float).reshape(-1,3)
    fs=len(spheres)*(4/3)*np.pi*r_soma**3/L**3 if r_soma>0 else 0.0
    fn=len(cylinders)*np.pi*r_neurite**2/L**2
    return Substrate(L,r_neurite,cylinders,spheres,float(r_soma),substrate_type),fs,fn

## Brownian walker simulation with reflecting boundaries

For each timestep, each Cartesian Brownian proposal satisfies

$$\Delta x_i\sim N(0,2D_0\Delta t).$$

Instead of rejecting an entire step when its endpoint enters an obstacle, the code now:

1. finds the first segment-surface intersection,
2. moves the walker to the collision point,
3. reflects the remaining displacement about the local outward normal,
4. continues the reflected remainder,
5. allows multiple collisions within one timestep.

Wrapped coordinates are used for periodic collision geometry while cumulative unwrapped displacement is retained for MSD.

The collision diagnostics are:

- `collision_step_fraction`: fraction of walker-steps with at least one collision;
- `mean_collisions_per_step`: average number of surface collisions per walker-step;
- `truncated_step_fraction`: fraction of walker-steps that hit the maximum collision safeguard. This should be essentially zero.

In [4]:
def initialise_walkers(sub, n, rng, batch=10000):
    parts = []
    count = 0
    while count < n:
        x = rng.uniform(0, sub.box_length, size=(max(batch, n-count), 3))
        x = x[~inside_obstacles(x, sub)]
        take = min(len(x), n-count)
        if take:
            parts.append(x[:take])
            count += take
    return np.vstack(parts)


COLLISION_EPS = 1e-10


def first_sphere_hit(start, step, centers, radius, L):
    """First intersection of a segment with any periodic sphere."""
    if len(centers) == 0:
        return None, None

    rel = periodic_delta(start[None, :] - centers, L)
    a = float(np.dot(step, step))
    if a <= 0:
        return None, None

    b = 2.0 * (rel @ step)
    c = np.sum(rel*rel, axis=1) - radius**2
    disc = b*b - 4.0*a*c
    valid = disc >= 0
    if not np.any(valid):
        return None, None

    idx = np.where(valid)[0]
    root = np.sqrt(np.maximum(disc[idx], 0.0))
    l1 = (-b[idx] - root) / (2.0*a)
    l2 = (-b[idx] + root) / (2.0*a)

    candidates = []
    for j, obj_i in enumerate(idx):
        for lam in (l1[j], l2[j]):
            if COLLISION_EPS < lam <= 1.0 + COLLISION_EPS:
                candidates.append((float(lam), int(obj_i)))
    if not candidates:
        return None, None

    lam, obj_i = min(candidates, key=lambda z: z[0])
    radial = rel[obj_i] + lam*step
    nr = np.linalg.norm(radial)
    if nr == 0:
        return None, None
    return lam, radial/nr


def first_cylinder_hit(start, step, centers_xy, radius, L):
    """First intersection with any periodic infinite z-aligned cylinder."""
    if len(centers_xy) == 0:
        return None, None

    step_xy = step[:2]
    a = float(np.dot(step_xy, step_xy))
    if a <= 0:
        return None, None

    rel = periodic_delta(start[None, :2] - centers_xy, L)
    b = 2.0 * (rel @ step_xy)
    c = np.sum(rel*rel, axis=1) - radius**2
    disc = b*b - 4.0*a*c
    valid = disc >= 0
    if not np.any(valid):
        return None, None

    idx = np.where(valid)[0]
    root = np.sqrt(np.maximum(disc[idx], 0.0))
    l1 = (-b[idx] - root) / (2.0*a)
    l2 = (-b[idx] + root) / (2.0*a)

    candidates = []
    for j, obj_i in enumerate(idx):
        for lam in (l1[j], l2[j]):
            if COLLISION_EPS < lam <= 1.0 + COLLISION_EPS:
                candidates.append((float(lam), int(obj_i)))
    if not candidates:
        return None, None

    lam, obj_i = min(candidates, key=lambda z: z[0])
    radial_xy = rel[obj_i] + lam*step_xy
    nr = np.linalg.norm(radial_xy)
    if nr == 0:
        return None, None
    return lam, np.array([radial_xy[0]/nr, radial_xy[1]/nr, 0.0])


def first_obstacle_hit(start, step, sub):
    ls, ns = first_sphere_hit(start, step, sub.sphere_xyz, sub.sphere_r, sub.box_length)
    lc, nc = first_cylinder_hit(start, step, sub.cylinder_xy, sub.r_neurite, sub.box_length)
    hits = []
    if ls is not None:
        hits.append((ls, ns, "sphere"))
    if lc is not None:
        hits.append((lc, nc, "cylinder"))
    return min(hits, key=lambda z: z[0]) if hits else (None, None, None)


def propagate_reflecting_step(start, step, sub, *, max_collisions=8, surface_epsilon=1e-9):
    """Propagate one finite Brownian proposal with specular reflecting obstacles."""
    L = sub.box_length
    current = np.asarray(start, float).copy()
    remaining = np.asarray(step, float).copy()
    total = np.zeros(3, float)
    n_collisions = 0

    for _ in range(max_collisions):
        current_wrapped = np.mod(current, L)
        lam, normal, _kind = first_obstacle_hit(current_wrapped, remaining, sub)

        if lam is None:
            current += remaining
            total += remaining
            return np.mod(current, L), total, n_collisions, False

        lam = float(np.clip(lam, 0.0, 1.0))
        travelled = lam * remaining
        current += travelled
        total += travelled

        leftover = (1.0 - lam) * remaining
        reflected = leftover - 2.0*np.dot(leftover, normal)*normal

        n_collisions += 1
        current += surface_epsilon * normal
        total += surface_epsilon * normal
        remaining = reflected

        if np.dot(remaining, remaining) < 1e-24:
            return np.mod(current, L), total, n_collisions, False

    return np.mod(current, L), total, n_collisions, True


def simulate_diffusion(
    sub,
    *,
    D0=2.2,
    t_max=12.0,
    dt=0.01,
    n_walkers=10000,
    save_times=(1,2,4,8,12),
    seed=0,
    max_collisions_per_step=8,
):
    """Brownian diffusion with finite-step specular reflecting boundaries."""
    rng = np.random.default_rng(seed)
    wrapped = initialise_walkers(sub, n_walkers, rng)
    disp = np.zeros_like(wrapped)

    n_steps = int(round(t_max/dt))
    dt = t_max/n_steps
    sd = np.sqrt(2.0*D0*dt)
    save_steps = {int(round(t/dt)): t for t in save_times if 0 < t <= t_max}

    rows = []
    total_collisions = 0
    collision_steps = 0
    truncated_steps = 0

    for step_index in range(1, n_steps+1):
        proposed_steps = rng.normal(0.0, sd, size=wrapped.shape)

        for wi in range(n_walkers):
            new_pos, actual_step, nc, truncated = propagate_reflecting_step(
                wrapped[wi], proposed_steps[wi], sub,
                max_collisions=max_collisions_per_step,
            )
            wrapped[wi] = new_pos
            disp[wi] += actual_step
            total_collisions += nc
            collision_steps += int(nc > 0)
            truncated_steps += int(truncated)

        if step_index in save_steps:
            t = step_index*dt
            dx2 = disp[:,0]**2
            dy2 = disp[:,1]**2
            dz2 = disp[:,2]**2
            pc = dz2/(2.0*t)
            qc = (dx2+dy2)/(4.0*t)
            total_walker_steps = step_index*n_walkers
            rows.append(dict(
                time_ms=t,
                D_parallel=pc.mean(),
                D_perp=qc.mean(),
                SE_parallel=pc.std(ddof=1)/np.sqrt(n_walkers),
                SE_perp=qc.std(ddof=1)/np.sqrt(n_walkers),
                collision_step_fraction=collision_steps/total_walker_steps,
                mean_collisions_per_step=total_collisions/total_walker_steps,
                truncated_step_fraction=truncated_steps/total_walker_steps,
                dt_ms=dt,
                n_walkers=n_walkers,
            ))

    return pd.DataFrame(rows)

## Validation and convergence sequence

Run these in order before production:

1. free-diffusion unit test: both diffusivities should be approximately $D_0$;
2. soma-only unit test: $D_\parallel$ and $D_\perp$ should be approximately equal;
3. neurite-only unit test: $D_\parallel$ should remain approximately $D_0$, while $D_\perp<D_0$;
4. low mixed-substrate timestep x walker convergence with repeated walker seeds;
5. after the separate packing engine is fixed, repeat low/mid/high convergence, box-size convergence, and geometry-seed convergence;
6. only then enable production.

The current random-sequential packing code is intentionally left unchanged in this notebook so that collision and packing changes are not confounded.

In [13]:
# =============================================================================
# UNIT TESTS FOR THE REFLECTING COLLISION ENGINE
# =============================================================================

RUN_UNIT_TESTS = True

if RUN_UNIT_TESTS:
    # 1) Free diffusion: D_parallel ~= D_perp ~= D0
    empty_sub = Substrate(
        box_length=40.0,
        r_neurite=R_NEURITE,
        cylinder_xy=np.empty((0,2)),
        sphere_xyz=np.empty((0,3)),
        sphere_r=5.0,
        substrate_type="free",
    )
    free_test = simulate_diffusion(
        empty_sub, D0=D0, t_max=12.0, dt=0.01,
        n_walkers=5000, save_times=[12], seed=BASE_SEED,
    )
    print("FREE DIFFUSION")
    display(free_test)

    # 2) Soma-only: isotropic effective diffusion
    soma_sub, fs_soma, fn_soma = make_substrate(
        0.10, 0.0, 5.0, box_length=40.0,
        r_neurite=R_NEURITE, seed=BASE_SEED+1,
        substrate_type="soma_only",
    )
    soma_test = simulate_diffusion(
        soma_sub, D0=D0, t_max=12.0, dt=0.01,
        n_walkers=5000, save_times=[12], seed=BASE_SEED+2,
    )
    print("SOMA ONLY", "fs_real=", fs_soma)
    display(soma_test)

    # 3) Neurite-only: D_parallel ~= D0; D_perp < D0
    neur_sub, fs_neur, fn_neur = make_substrate(
        0.0, 0.20, 5.0, box_length=40.0,
        r_neurite=R_NEURITE, seed=BASE_SEED+3,
        substrate_type="neurite_only",
    )
    neur_test = simulate_diffusion(
        neur_sub, D0=D0, t_max=12.0, dt=0.01,
        n_walkers=5000, save_times=[12], seed=BASE_SEED+4,
    )
    print("NEURITE ONLY", "fn_real=", fn_neur)
    display(neur_test)

    print("\nChecks:")
    print("free Dpar/D0 =", float(free_test.iloc[-1].D_parallel/D0))
    print("free Dperp/D0 =", float(free_test.iloc[-1].D_perp/D0))
    print("soma anisotropy Dpar-Dperp =", float(soma_test.iloc[-1].D_parallel-soma_test.iloc[-1].D_perp))
    print("neurite Dpar/D0 =", float(neur_test.iloc[-1].D_parallel/D0))
    print("neurite Dperp/D0 =", float(neur_test.iloc[-1].D_perp/D0))
else:
    print("Set RUN_UNIT_TESTS=True first.")

FREE DIFFUSION


,time_ms,D_parallel,D_perp,SE_parallel,SE_perp,collision_step_fraction,mean_collisions_per_step,truncated_step_fraction,dt_ms,n_walkers
0,12.0,2.211899,2.162244,0.04483,0.030546,0.0,0.0,0.0,0.01,5000


SOMA ONLY fs_real= 0.09817477042468103


,time_ms,D_parallel,D_perp,SE_parallel,SE_perp,collision_step_fraction,mean_collisions_per_step,truncated_step_fraction,dt_ms,n_walkers
0,12.0,2.180988,2.151315,0.043349,0.030986,0.005263,0.005263,0.0,0.01,5000


NEURITE ONLY fn_real= 0.19978565781422591


,time_ms,D_parallel,D_perp,SE_parallel,SE_perp,collision_step_fraction,mean_collisions_per_step,truncated_step_fraction,dt_ms,n_walkers
0,12.0,2.151109,1.802705,0.043151,0.025439,0.081273,0.083874,0.000018,0.01,5000



Checks:
free Dpar/D0 = 1.0054086798434978
free Dperp/D0 = 0.982838095452615
soma anisotropy Dpar-Dperp = 0.029672896435328777
neurite Dpar/D0 = 0.9777766030399713
neurite Dperp/D0 = 0.8194112420728616


In [14]:
neur_test_20k = simulate_diffusion(
    neur_sub,
    D0=D0,
    t_max=12.0,
    dt=0.01,
    n_walkers=20000,
    save_times=[12],
    seed=BASE_SEED + 100,
)

display(neur_test_20k)

print(
    "Dpar/D0 =",
    neur_test_20k.iloc[-1].D_parallel / D0
)

print(
    "Dperp/D0 =",
    neur_test_20k.iloc[-1].D_perp / D0
)

,time_ms,D_parallel,D_perp,SE_parallel,SE_perp,collision_step_fraction,mean_collisions_per_step,truncated_step_fraction,dt_ms,n_walkers
0,12.0,2.18112,1.762098,0.021789,0.012646,0.08079,0.083498,0.000018,0.01,20000


Dpar/D0 = 0.9914181705471611
Dperp/D0 = 0.800953525262078


In [15]:
# =============================================================================
# REFLECTING-BOUNDARY CONVERGENCE EXPERIMENT
# =============================================================================
#
# PURPOSE
# -------
# Jointly test:
#
#   1. timestep convergence
#   2. walker-number convergence
#   3. Monte-Carlo variability across independent repeats
#   4. whether convergence differs with obstruction level
#
# Box size is deliberately FIXED at 40 um here.
# Box-size convergence will be tested separately once dt and walker
# count have been selected.
#
# IMPORTANT
# ---------
# - Results are checkpointed after EVERY simulation.
# - The experiment can therefore be safely stopped and resumed.
# - Geometry is fixed within each tissue condition.
# - Brownian walker seeds vary between repeats.
# =============================================================================

from pathlib import Path
import time
import numpy as np
import pandas as pd


# =============================================================================
# EXPERIMENT SETTINGS
# =============================================================================

CONVERGENCE_CASES = [

    (
        "low",
        0.05,   # f_soma
        0.20,   # f_neurite
        5.0,    # r_soma
    ),

    (
        "medium",
        0.15,
        0.30,
        5.0,
    ),

    (
        "high",
        0.25,
        0.40,
        5.0,
    ),

]


# Timestep grid
DT_VALUES = [

    0.020,
    0.010,
    0.005,
    0.0025,

]


# Walker-number grid
WALKER_VALUES = [

    5000,
    10000,
    20000,
    40000,

]


# Independent Brownian repeats
N_REPEATS = 4


# Keep box size fixed during this experiment
BOX_LENGTH = 40.0


RUN_CONVERGENCE = True


# =============================================================================
# CHECKPOINT FILE
# =============================================================================

CHECKPOINT_FILE = (
    OUTDIR
    / "reflecting_convergence_checkpoint.csv"
)


# =============================================================================
# LOAD PREVIOUS RESULTS IF THEY EXIST
# =============================================================================

if CHECKPOINT_FILE.exists():

    convergence_df = pd.read_csv(
        CHECKPOINT_FILE
    )

    conv = convergence_df.to_dict(
        orient="records"
    )

    print(
        f"Loaded {len(conv)} "
        "previous checkpoint rows."
    )

else:

    conv = []

    convergence_df = pd.DataFrame()

    print(
        "No previous checkpoint found."
    )


# =============================================================================
# IDENTIFY COMPLETED SIMULATIONS
# =============================================================================

completed = set()


for row in conv:

    # Only successful simulations count as completed.
    if (
        row.get("status")
        == "success"
    ):

        completed.add(

            (
                str(
                    row["case"]
                ),

                float(
                    row["box_length"]
                ),

                float(
                    row["dt_ms"]
                ),

                int(
                    row["n_walkers"]
                ),

                int(
                    row["repeat"]
                ),

            )

        )


print(
    "Successful simulations already completed:",
    len(completed),
)


# =============================================================================
# COUNT TOTAL REQUESTED RUNS
# =============================================================================

total_requested = (

    len(CONVERGENCE_CASES)
    * len(DT_VALUES)
    * len(WALKER_VALUES)
    * N_REPEATS

)


print(
    "Maximum requested simulations:",
    total_requested,
)


# =============================================================================
# RUN EXPERIMENT
# =============================================================================

if RUN_CONVERGENCE:

    experiment_start = time.time()

    new_successes = 0

    new_failures = 0


    # =========================================================================
    # LOOP OVER BIOLOGICAL CONDITIONS
    # =========================================================================

    for ci, (
        name,
        fs,
        fn,
        rs,
    ) in enumerate(
        CONVERGENCE_CASES
    ):


        print(
            "\n"
            + "=" * 75
        )

        print(
            f"CASE: {name}"
        )

        print(
            f"Target f_soma    = {fs:.4f}"
        )

        print(
            f"Target f_neurite = {fn:.4f}"
        )

        print(
            f"Soma radius      = {rs:.2f} um"
        )

        print(
            f"Box length       = {BOX_LENGTH:.1f} um"
        )

        print(
            "=" * 75
        )


        # =====================================================================
        # FIXED GEOMETRY SEED
        # =====================================================================
        #
        # Every numerical setting for this biological case uses the SAME
        # substrate geometry.
        #
        # This is important:
        #
        # differences between dt / walker count / repeat should therefore
        # arise from numerical simulation rather than changing obstacle
        # arrangement.
        # =====================================================================

        geometry_seed = (

            BASE_SEED
            + ci * 10000
            + int(
                BOX_LENGTH
            )

        )


        # =====================================================================
        # BUILD GEOMETRY ONCE
        # =====================================================================

        try:

            sub, fs_real, fn_real = (
                make_substrate(

                    fs,
                    fn,
                    rs,

                    box_length=
                        BOX_LENGTH,

                    r_neurite=
                        R_NEURITE,

                    seed=
                        geometry_seed,

                    substrate_type=
                        "convergence",

                )
            )


            print(
                "\nGeometry successfully generated."
            )

            print(
                f"f_soma:    "
                f"target={fs:.5f}, "
                f"realised={fs_real:.5f}"
            )

            print(
                f"f_neurite: "
                f"target={fn:.5f}, "
                f"realised={fn_real:.5f}"
            )

            print(
                f"f_extra:   "
                f"{1.0 - fs_real - fn_real:.5f}"
            )


        except Exception as exc:

            print(
                "\nGEOMETRY PACKING FAILED:"
            )

            print(
                repr(exc)
            )


            # -------------------------------------------------------------
            # Save ONE geometry-failure row.
            # -------------------------------------------------------------

            failure_row = {

                "case":
                    name,

                "status":
                    "geometry_failed",

                "error":
                    repr(exc),

                "geometry_seed":
                    geometry_seed,

                "box_length":
                    BOX_LENGTH,

                "f_soma_target":
                    fs,

                "f_neurite_target":
                    fn,

                "r_soma":
                    rs,

            }


            conv.append(
                failure_row
            )


            pd.DataFrame(
                conv
            ).to_csv(
                CHECKPOINT_FILE,
                index=False,
            )


            # Don't try numerical simulations
            # for a geometry that doesn't exist.
            continue


        # =====================================================================
        # LOOP OVER NUMERICAL SETTINGS
        # =====================================================================

        for dt in DT_VALUES:

            for nw in WALKER_VALUES:

                for repeat in range(
                    N_REPEATS
                ):


                    # =========================================================
                    # UNIQUE RUN KEY
                    # =========================================================

                    key = (

                        name,

                        float(
                            BOX_LENGTH
                        ),

                        float(
                            dt
                        ),

                        int(
                            nw
                        ),

                        int(
                            repeat
                        ),

                    )


                    # =========================================================
                    # SKIP COMPLETED RUNS
                    # =========================================================

                    if key in completed:

                        print(
                            "Skipping completed:",
                            key,
                        )

                        continue


                    # =========================================================
                    # WALKER SEED
                    # =========================================================
                    #
                    # Deterministic but unique for each condition.
                    #
                    # Geometry remains unchanged.
                    # =========================================================

                    walker_seed = (

                        BASE_SEED
                        + 1000000

                        + ci
                        * 100000

                        + int(
                            round(
                                dt
                                * 1_000_000
                            )
                        )
                        * 10

                        + int(
                            nw
                        )

                        + repeat
                        * 1000

                    )


                    print(
                        "\n"
                        f"Running "
                        f"{name}: "
                        f"dt={dt:g} ms, "
                        f"N={nw:,}, "
                        f"repeat="
                        f"{repeat + 1}/{N_REPEATS}"
                    )


                    t0 = time.time()


                    # =========================================================
                    # RUN REFLECTING-BOUNDARY DIFFUSION
                    # =========================================================

                    try:

                        sim = (
                            simulate_diffusion(

                                sub,

                                D0=D0,

                                t_max=T_MAX,

                                dt=dt,

                                n_walkers=nw,

                                save_times=[
                                    12.0
                                ],

                                seed=
                                    walker_seed,

                            )
                        )


                        result = (
                            sim.iloc[-1]
                            .to_dict()
                        )


                        runtime = (
                            time.time()
                            - t0
                        )


                        # =====================================================
                        # STORE RESULT
                        # =====================================================

                        row = {

                            **result,

                            "case":
                                name,

                            "status":
                                "success",

                            "repeat":
                                repeat,

                            "geometry_seed":
                                geometry_seed,

                            "walker_seed":
                                walker_seed,

                            "box_length":
                                BOX_LENGTH,

                            "f_soma_target":
                                fs,

                            "f_neurite_target":
                                fn,

                            "f_soma_real":
                                fs_real,

                            "f_neurite_real":
                                fn_real,

                            "f_extra_real":
                                (
                                    1.0
                                    - fs_real
                                    - fn_real
                                ),

                            "r_soma":
                                rs,

                            "runtime_s":
                                runtime,

                        }


                        conv.append(
                            row
                        )


                        completed.add(
                            key
                        )


                        new_successes += 1


                        # =====================================================
                        # CHECKPOINT IMMEDIATELY
                        # =====================================================

                        pd.DataFrame(
                            conv
                        ).to_csv(
                            CHECKPOINT_FILE,
                            index=False,
                        )


                        # =====================================================
                        # PRINT USEFUL SUMMARY
                        # =====================================================

                        print(
                            f"  D_parallel = "
                            f"{row['D_parallel']:.6f}"
                        )

                        print(
                            f"  D_perp     = "
                            f"{row['D_perp']:.6f}"
                        )

                        print(
                            f"  collisions = "
                            f"{100 * row['collision_step_fraction']:.3f}%"
                        )

                        print(
                            f"  truncated  = "
                            f"{100 * row['truncated_step_fraction']:.6f}%"
                        )

                        print(
                            f"  runtime    = "
                            f"{runtime:.1f} s"
                        )


                    # =========================================================
                    # SIMULATION FAILURE
                    # =========================================================

                    except Exception as exc:

                        runtime = (
                            time.time()
                            - t0
                        )


                        print(
                            "\nSIMULATION FAILED:"
                        )

                        print(
                            repr(exc)
                        )


                        failure_row = {

                            "case":
                                name,

                            "status":
                                "simulation_failed",

                            "error":
                                repr(exc),

                            "repeat":
                                repeat,

                            "geometry_seed":
                                geometry_seed,

                            "walker_seed":
                                walker_seed,

                            "box_length":
                                BOX_LENGTH,

                            "dt_ms":
                                dt,

                            "n_walkers":
                                nw,

                            "f_soma_target":
                                fs,

                            "f_neurite_target":
                                fn,

                            "f_soma_real":
                                fs_real,

                            "f_neurite_real":
                                fn_real,

                            "r_soma":
                                rs,

                            "runtime_s":
                                runtime,

                        }


                        conv.append(
                            failure_row
                        )


                        new_failures += 1


                        # Checkpoint failure too.
                        pd.DataFrame(
                            conv
                        ).to_csv(
                            CHECKPOINT_FILE,
                            index=False,
                        )


    # =========================================================================
    # FINAL DATAFRAME
    # =========================================================================

    convergence_df = pd.DataFrame(
        conv
    )


    elapsed = (
        time.time()
        - experiment_start
    )


    print(
        "\n"
        + "=" * 75
    )

    print(
        "CONVERGENCE EXPERIMENT FINISHED"
    )

    print(
        "=" * 75
    )

    print(
        "New successful runs:",
        new_successes,
    )

    print(
        "New failed runs:",
        new_failures,
    )

    print(
        "Elapsed time:",
        f"{elapsed / 3600:.2f} hours",
    )

    print(
        "Checkpoint:",
        CHECKPOINT_FILE,
    )


    display(
        convergence_df
    )


else:

    print(
        "Set RUN_CONVERGENCE=True "
        "after unit tests pass."
    )

No previous checkpoint found.
Successful simulations already completed: 0
Maximum requested simulations: 192

CASE: low
Target f_soma    = 0.0500
Target f_neurite = 0.2000
Soma radius      = 5.00 um
Box length       = 40.0 um

Geometry successfully generated.
f_soma:    target=0.05000, realised=0.04909
f_neurite: target=0.20000, realised=0.19979
f_extra:   0.75113

Running low: dt=0.02 ms, N=5,000, repeat=1/4
  D_parallel = 2.125161
  D_perp     = 1.666934
  collisions = 11.974%
  truncated  = 0.006467%
  runtime    = 111.2 s

Running low: dt=0.02 ms, N=5,000, repeat=2/4
  D_parallel = 2.131269
  D_perp     = 1.683162
  collisions = 12.028%
  truncated  = 0.006500%
  runtime    = 111.0 s

Running low: dt=0.02 ms, N=5,000, repeat=3/4
  D_parallel = 2.198922
  D_perp     = 1.684386
  collisions = 11.938%
  truncated  = 0.005533%
  runtime    = 109.3 s

Running low: dt=0.02 ms, N=5,000, repeat=4/4
  D_parallel = 2.139616
  D_perp     = 1.712873
  collisions = 12.060%
  truncated  = 0.0054

,time_ms,D_parallel,D_perp,SE_parallel,SE_perp,collision_step_fraction,mean_collisions_per_step,truncated_step_fraction,dt_ms,n_walkers,...,walker_seed,box_length,f_soma_target,f_neurite_target,f_soma_real,f_neurite_real,f_extra_real,r_soma,runtime_s,error
0,12.0,2.125161,1.666934,0.042570,0.023951,0.119739,0.129207,0.000065,0.0200,5000.0,...,1217345.0,40.0,0.05,0.2,0.049087,0.199786,0.751127,5.0,111.184976,NaN
1,12.0,2.131269,1.683162,0.044665,0.023715,0.120282,0.129829,0.000065,0.0200,5000.0,...,1218345.0,40.0,0.05,0.2,0.049087,0.199786,0.751127,5.0,110.955275,NaN
2,12.0,2.198922,1.684386,0.044174,0.023814,0.119376,0.128740,0.000055,0.0200,5000.0,...,1219345.0,40.0,0.05,0.2,0.049087,0.199786,0.751127,5.0,109.285537,NaN
3,12.0,2.139616,1.712873,0.042970,0.024201,0.120599,0.130311,0.000054,0.0200,5000.0,...,1220345.0,40.0,0.05,0.2,0.049087,0.199786,0.751127,5.0,109.183787,NaN
4,12.0,2.133821,1.682398,0.029995,0.017151,0.119891,0.129401,0.000056,0.0200,10000.0,...,1222345.0,40.0,0.05,0.2,0.049087,0.199786,0.751127,5.0,218.641763,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,12.0,2.155623,1.742259,0.015412,0.008873,0.045329,0.046228,0.000007,0.0025,40000.0,...,1078345.0,40.0,0.05,0.2,0.049087,0.199786,0.751127,5.0,6301.654621,NaN
62,12.0,2.178471,1.752756,0.015482,0.008894,0.045044,0.045925,0.000007,0.0025,40000.0,...,1079345.0,40.0,0.05,0.2,0.049087,0.199786,0.751127,5.0,6324.913184,NaN
63,12.0,2.116369,1.731829,0.015032,0.008845,0.045471,0.046374,0.000007,0.0025,40000.0,...,1080345.0,40.0,0.05,0.2,0.049087,0.199786,0.751127,5.0,6308.167804,NaN
64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,40.0,0.15,0.3,NaN,NaN,NaN,5.0,NaN,RuntimeError('Cylinder packing failed 275/611')


In [16]:
# =============================================================================
# SUMMARISE SUCCESSFUL CONVERGENCE RUNS
# =============================================================================

successful = (
    convergence_df[
        convergence_df["status"]
        == "success"
    ]
    .copy()
)


if len(successful) == 0:

    print(
        "No successful simulations available."
    )

else:

    convergence_summary = (

        successful

        .groupby(
            [
                "case",
                "dt_ms",
                "n_walkers",
            ]
        )

        .agg(

            n_repeats=(
                "D_parallel",
                "size",
            ),

            D_parallel_mean=(
                "D_parallel",
                "mean",
            ),

            D_parallel_sd=(
                "D_parallel",
                "std",
            ),

            D_parallel_sem=(
                "D_parallel",
                lambda x:
                    x.std(ddof=1)
                    / np.sqrt(
                        len(x)
                    ),
            ),

            D_perp_mean=(
                "D_perp",
                "mean",
            ),

            D_perp_sd=(
                "D_perp",
                "std",
            ),

            D_perp_sem=(
                "D_perp",
                lambda x:
                    x.std(ddof=1)
                    / np.sqrt(
                        len(x)
                    ),
            ),

            collision_fraction_mean=(
                "collision_step_fraction",
                "mean",
            ),

            mean_collisions_per_step=(
                "mean_collisions_per_step",
                "mean",
            ),

            truncated_fraction_mean=(
                "truncated_step_fraction",
                "mean",
            ),

            runtime_mean_s=(
                "runtime_s",
                "mean",
            ),

        )

        .reset_index()

        .sort_values(
            [
                "case",
                "dt_ms",
                "n_walkers",
            ]
        )

        .reset_index(
            drop=True
        )
    )


    display(
        convergence_summary
    )


    convergence_summary.to_csv(
        OUTDIR
        / "reflecting_convergence_summary.csv",
        index=False,
    )

,case,dt_ms,n_walkers,n_repeats,D_parallel_mean,D_parallel_sd,D_parallel_sem,D_perp_mean,D_perp_sd,D_perp_sem,collision_fraction_mean,mean_collisions_per_step,truncated_fraction_mean,runtime_mean_s
0,low,0.0025,5000.0,4,2.133260,0.037968,0.018984,1.759863,0.022669,0.011335,0.045033,0.045917,0.000007,790.351819
1,low,0.0025,10000.0,4,2.141787,0.038828,0.019414,1.740201,0.010996,0.005498,0.045090,0.045969,0.000007,1578.483950
2,low,0.0025,20000.0,4,2.146002,0.008914,0.004457,1.746775,0.007080,0.003540,0.045210,0.046083,0.000007,3169.075235
3,low,0.0025,40000.0,4,2.148769,0.025795,0.012898,1.743105,0.008701,0.004351,0.045294,0.046189,0.000007,6317.067136
4,low,0.0050,5000.0,4,2.119053,0.032431,0.016216,1.718934,0.021055,0.010527,0.063092,0.065076,0.000015,394.709383
5,low,0.0050,10000.0,4,2.145405,0.036648,0.018324,1.730036,0.009707,0.004853,0.063155,0.065179,0.000015,788.911948
6,low,0.0050,20000.0,4,2.123406,0.061817,0.030909,1.739893,0.014420,0.007210,0.063342,0.065347,0.000015,1579.612825
7,low,0.0050,40000.0,4,2.134064,0.007777,0.003889,1.734435,0.013369,0.006684,0.063301,0.065316,0.000015,3202.569149
8,low,0.0100,5000.0,4,2.095165,0.030637,0.015318,1.711563,0.022911,0.011455,0.088246,0.092754,0.000030,209.595921
9,low,0.0100,10000.0,4,2.151233,0.025377,0.012689,1.735018,0.014997,0.007498,0.087831,0.092299,0.000031,414.816616


In [7]:
# =============================================================================
# SUMMARISE CONVERGENCE REPEATS
# =============================================================================

if "convergence_df" in globals() and len(convergence_df):
    convergence_summary = (
        convergence_df
        .groupby(["dt_ms", "n_walkers"])
        .agg(
            n_repeats=("D_parallel", "size"),
            D_parallel_mean=("D_parallel", "mean"),
            D_parallel_sd=("D_parallel", "std"),
            D_perp_mean=("D_perp", "mean"),
            D_perp_sd=("D_perp", "std"),
            collision_fraction_mean=("collision_step_fraction", "mean"),
            mean_collisions_mean=("mean_collisions_per_step", "mean"),
            truncated_fraction_mean=("truncated_step_fraction", "mean"),
            runtime_mean_s=("runtime_s", "mean"),
        )
        .reset_index()
    )
    display(convergence_summary)

    # Percentage change when halving dt at each walker count.
    change_rows = []
    for nw in sorted(convergence_summary.n_walkers.unique()):
        q = convergence_summary[convergence_summary.n_walkers == nw].sort_values("dt_ms", ascending=False).reset_index(drop=True)
        for i in range(len(q)-1):
            coarse, fine = q.iloc[i], q.iloc[i+1]
            change_rows.append(dict(
                n_walkers=nw,
                dt_coarse=coarse.dt_ms,
                dt_fine=fine.dt_ms,
                parallel_change_percent=100*abs(fine.D_parallel_mean-coarse.D_parallel_mean)/abs(fine.D_parallel_mean),
                perp_change_percent=100*abs(fine.D_perp_mean-coarse.D_perp_mean)/abs(fine.D_perp_mean),
            ))
    dt_change_df = pd.DataFrame(change_rows)
    display(dt_change_df)
else:
    print("No convergence results yet.")

No convergence results yet.


In [8]:
# =============================================================================
# CONVERGENCE PLOTS
# =============================================================================

if "convergence_summary" in globals() and len(convergence_summary):
    for metric, sdcol, ylabel in [
        ("D_parallel_mean", "D_parallel_sd", r"$D_{ex\parallel}$ ($\mu m^2/ms$)"),
        ("D_perp_mean", "D_perp_sd", r"$D_{ex\perp}$ ($\mu m^2/ms$)"),
    ]:
        fig, ax = plt.subplots(figsize=(7,5))
        for nw in sorted(convergence_summary.n_walkers.unique()):
            q = convergence_summary[convergence_summary.n_walkers == nw].sort_values("dt_ms")
            ax.errorbar(q.dt_ms, q[metric], yerr=q[sdcol], marker="o", capsize=3, label=f"N={int(nw)}")
        ax.set_xlabel(r"$\Delta t$ (ms)")
        ax.set_ylabel(ylabel)
        ax.set_title("Reflecting-boundary timestep convergence")
        ax.invert_xaxis()
        ax.legend()
        plt.show()

    fig, ax = plt.subplots(figsize=(7,5))
    for nw in sorted(convergence_summary.n_walkers.unique()):
        q = convergence_summary[convergence_summary.n_walkers == nw].sort_values("dt_ms")
        ax.plot(q.dt_ms, q.truncated_fraction_mean, marker="o", label=f"N={int(nw)}")
    ax.set_xlabel(r"$\Delta t$ (ms)")
    ax.set_ylabel("Truncated walker-step fraction")
    ax.set_title("Collision safeguard diagnostic")
    ax.invert_xaxis()
    ax.legend()
    plt.show()
else:
    print("No convergence summary yet.")

No convergence summary yet.


In [9]:
# =============================================================================
# NEXT-STAGE NUMERICAL CHECKS
# =============================================================================
# Do not run production yet. After the reflecting-boundary low-case convergence
# is satisfactory, the next tasks are:
#   1) fix/replace the mixed-substrate packing engine;
#   2) rerun low/mid/high convergence;
#   3) test box size;
#   4) test independent geometry seeds.
#
# Production settings below are placeholders until those tests pass.

print("Collision engine updated. Production remains disabled pending convergence and packing validation.")

Collision engine updated. Production remains disabled pending convergence and packing validation.


In [10]:
# Placeholder production settings ONLY. Update after convergence is demonstrated.
PRODUCTION_DT = 0.005
PRODUCTION_N_WALKERS = 20000
PRODUCTION_BOX_LENGTH = 40.0

print("1D Brownian proposal SD =", np.sqrt(2*D0*PRODUCTION_DT), "um")
print("WARNING: production settings are provisional; RUN_PRODUCTION must remain False until validation passes.")

1D Brownian proposal SD = 0.14832396974191325 um


## Experimental design

144 mixed substrates + 36 soma-only controls + 4 neurite-only controls = 184 total. The controls isolate each obstruction mechanism before mixed-substrate validation.

In [11]:
rows=[]; sid=0
for fs in F_SOMA_VALUES:
    for fn in F_NEURITE_VALUES:
        for rs in R_SOMA_VALUES:
            rows.append(dict(substrate_id=sid,substrate_type="mixed",f_soma_target=float(fs),f_neurite_target=float(fn),r_soma=float(rs))); sid+=1
for fs in F_SOMA_VALUES:
    for rs in R_SOMA_VALUES:
        rows.append(dict(substrate_id=sid,substrate_type="soma_only",f_soma_target=float(fs),f_neurite_target=0.0,r_soma=float(rs))); sid+=1
for fn in F_NEURITE_VALUES:
    rows.append(dict(substrate_id=sid,substrate_type="neurite_only",f_soma_target=0.0,f_neurite_target=float(fn),r_soma=0.0)); sid+=1
design=pd.DataFrame(rows); design["f_extra_target"]=1-design.f_soma_target-design.f_neurite_target
print(design.substrate_type.value_counts()); print("Total",len(design)); display(design)

substrate_type
mixed           144
soma_only        36
neurite_only      4
Name: count, dtype: int64
Total 184


,substrate_id,substrate_type,f_soma_target,f_neurite_target,r_soma,f_extra_target
0,0,mixed,0.05,0.2,3.0,0.75
1,1,mixed,0.05,0.2,4.0,0.75
2,2,mixed,0.05,0.2,5.0,0.75
3,3,mixed,0.05,0.2,6.0,0.75
4,4,mixed,0.05,0.2,7.0,0.75
...,...,...,...,...,...,...
179,179,soma_only,0.30,0.0,8.0,0.70
180,180,neurite_only,0.00,0.2,0.0,0.80
181,181,neurite_only,0.00,0.3,0.0,0.70
182,182,neurite_only,0.00,0.4,0.0,0.60


## Production runner with checkpointing

**Do not enable production yet.** The reflecting collision engine must first pass the unit/convergence tests, and the current random-sequential packing algorithm must be replaced or validated for the middle/high mixed fractions. The previous packer failed for those cases.

Once both issues are resolved, this section can run the full 184-substrate design with checkpointing.

In [12]:
RESULTS_FILE=OUTDIR/"mc_substrate_results.csv"
RUN_PRODUCTION=False  # keep False until collision + packing validation pass
existing=pd.read_csv(RESULTS_FILE) if RESULTS_FILE.exists() else pd.DataFrame()
completed=set(existing.loc[existing.status=="ok","substrate_id"].astype(int)) if len(existing) else set()
out=existing.to_dict("records") if len(existing) else []
if RUN_PRODUCTION:
    for _,spec in design.iterrows():
        sid=int(spec.substrate_id)
        if sid in completed: continue
        print(f"[{sid+1}/{len(design)}] {spec.substrate_type} fs={spec.f_soma_target:.2f} fn={spec.f_neurite_target:.2f} rs={spec.r_soma:.1f}")
        t0=time.time()
        try:
            sub,fsr,fnr=make_substrate(spec.f_soma_target,spec.f_neurite_target,max(spec.r_soma,1e-9),box_length=PRODUCTION_BOX_LENGTH,r_neurite=R_NEURITE,seed=BASE_SEED+10*sid,substrate_type=spec.substrate_type)
            sim=simulate_diffusion(sub,D0=D0,t_max=T_MAX,dt=PRODUCTION_DT,n_walkers=PRODUCTION_N_WALKERS,save_times=SAVE_TIMES,seed=BASE_SEED+10*sid+1)
            runtime=time.time()-t0
            for _,s in sim.iterrows():
                out.append({**spec.to_dict(),"f_soma_real":fsr,"f_neurite_real":fnr,"f_extra_real":1-fsr-fnr,**s.to_dict(),"box_length":PRODUCTION_BOX_LENGTH,"runtime_s":runtime,"status":"ok"})
            completed.add(sid)
        except Exception as e:
            out.append({**spec.to_dict(),"status":"failed","error":repr(e),"runtime_s":time.time()-t0}); print("FAILED",e)
        pd.DataFrame(out).to_csv(RESULTS_FILE,index=False)
else: print("Set RUN_PRODUCTION=True after convergence testing")

Set RUN_PRODUCTION=True after convergence testing


# Analysis: controls first, mixed validation second

Soma-only controls estimate $a$ from $D/D_0=(1-f_s)^a$. Neurite-only controls estimate $b$ from $D_\perp/D_\parallel=(1-f_n)^b$ and directly test whether $D_\parallel\approx D_0$. Mixed substrates are then an independent validation set for the multiplicative law.

In [ ]:
def slope0(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float); return np.sum(x*y)/np.sum(x*x)

if RESULTS_FILE.exists():
    results=pd.read_csv(RESULTS_FILE); ok=results[results.status=="ok"].copy(); final12=ok[np.isclose(ok.time_ms,12)].copy()
    soma=final12[final12.substrate_type=="soma_only"].copy(); neur=final12[final12.substrate_type=="neurite_only"].copy(); mixed=final12[final12.substrate_type=="mixed"].copy()
    soma["a_implied"]=np.log(soma.D_parallel/D0)/np.log(1-soma.f_soma_real)
    a_control=slope0(np.log(1-soma.f_soma_real),np.log(soma.D_parallel/D0))
    neur["b_implied"]=np.log(neur.D_perp/neur.D_parallel)/np.log(1-neur.f_neurite_real)
    b_control=slope0(np.log(1-neur.f_neurite_real),np.log(neur.D_perp/neur.D_parallel))
    print("a_control",a_control,"b_control",b_control)
    print("neurite-only Dpar/D0",(neur.D_parallel/D0).to_numpy())
    display(soma); display(neur)
else: print("No production results yet")

In [ ]:
if "mixed" in globals() and len(mixed):
    mixed["S"]=1-mixed.f_soma_real/(1-mixed.f_neurite_real)
    mixed["N"]=1-mixed.f_neurite_real/(1-mixed.f_soma_real)
    mixed["a_implied"]=np.log(mixed.D_parallel/D0)/np.log(mixed.S)
    mixed["b_implied"]=np.log(mixed.D_perp/mixed.D_parallel)/np.log(mixed.N)
    mixed["D_parallel_pred"]=D0*mixed.S**a_control
    mixed["D_perp_pred"]=mixed.D_parallel_pred*mixed.N**b_control
    a_mixed=slope0(np.log(mixed.S),np.log(mixed.D_parallel/D0)); b_mixed=slope0(np.log(mixed.N),np.log(mixed.D_perp/mixed.D_parallel))
    print("controls a,b",a_control,b_control); print("mixed descriptive a,b",a_mixed,b_mixed)
    print("validation RMSE par",np.sqrt(np.mean((mixed.D_parallel_pred-mixed.D_parallel)**2)))
    print("validation RMSE perp",np.sqrt(np.mean((mixed.D_perp_pred-mixed.D_perp)**2)))

In [ ]:
if "mixed" in globals() and len(mixed):
    for obs,pred,title in [("D_parallel","D_parallel_pred","Parallel"),("D_perp","D_perp_pred","Perpendicular")]:
        fig,ax=plt.subplots(figsize=(6,6)); ax.scatter(mixed[obs],mixed[pred],alpha=.65); lo=min(mixed[obs].min(),mixed[pred].min()); hi=max(mixed[obs].max(),mixed[pred].max()); ax.plot([lo,hi],[lo,hi],"--"); ax.set(xlabel="Monte Carlo",ylabel="Control-calibrated prediction",title=title); plt.show()

    a_rs=mixed.groupby("r_soma").a_implied.agg(["mean","std","median","count"]).reset_index(); display(a_rs)
    fig,ax=plt.subplots(figsize=(7,5)); ax.errorbar(a_rs.r_soma,a_rs["mean"],yerr=a_rs["std"],marker="o",capsize=3); ax.axhline(a_control,ls="--"); ax.set(xlabel="Soma radius (um)",ylabel="Implied a",title="a versus soma radius"); plt.show()

    b_fn=mixed.groupby("f_neurite_target").b_implied.agg(["mean","std","median"]).reset_index(); display(b_fn)
    fig,ax=plt.subplots(figsize=(7,5)); ax.errorbar(b_fn.f_neurite_target,b_fn["mean"],yerr=b_fn["std"],marker="o",capsize=3); ax.axhline(b_control,ls="--"); ax.set(xlabel="f_neurite",ylabel="Implied b",title="b versus neurite fraction"); plt.show()

## Diffusion-time dependence and neurite-radius robustness

The saved intermediate times allow a/b to be inspected versus diffusion time. After the main run, repeat a small representative subset at neurite radii 0.25 and 0.4 um rather than immediately rerunning all 184 substrates.

In [ ]:
if "ok" in globals() and len(ok):
    tr=[]
    for t,q in ok.groupby("time_ms"):
        m=q[q.substrate_type=="mixed"].copy()
        if len(m):
            S=1-m.f_soma_real/(1-m.f_neurite_real); N=1-m.f_neurite_real/(1-m.f_soma_real)
            tr.append(dict(time_ms=t,a=slope0(np.log(S),np.log(m.D_parallel/D0)),b=slope0(np.log(N),np.log(m.D_perp/m.D_parallel))))
    exponent_time=pd.DataFrame(tr); display(exponent_time)
    fig,ax=plt.subplots(figsize=(7,5)); ax.plot(exponent_time.time_ms,exponent_time.a,marker="o",label="a"); ax.plot(exponent_time.time_ms,exponent_time.b,marker="o",label="b"); ax.set(xlabel="Diffusion time (ms)",ylabel="Global implied exponent",title="Time dependence"); ax.legend(); plt.show()

NEURITE_RADIUS_ROBUSTNESS=[0.25,0.40,0.50]
ROBUSTNESS_CASES=pd.DataFrame([
    {"f_soma":0.05,"f_neurite":0.20,"r_soma":5.0},
    {"f_soma":0.15,"f_neurite":0.30,"r_soma":5.0},
    {"f_soma":0.25,"f_neurite":0.40,"r_soma":5.0},
    {"f_soma":0.30,"f_neurite":0.50,"r_soma":5.0},
]); display(ROBUSTNESS_CASES)

# Interpretation checklist

1. Run the free-diffusion unit test: $D_\parallel\approx D_\perp\approx D_0$.
2. Run soma-only: verify approximate isotropy.
3. Run neurite-only: verify $D_\parallel\approx D_0$ and $D_\perp<D_0$.
4. Confirm `truncated_step_fraction` is essentially zero.
5. Establish timestep and walker convergence for the low mixed substrate using repeated walker seeds.
6. Fix/replace the mixed-substrate packing engine before middle/high tests.
7. Repeat convergence on low/mid/high obstruction cases.
8. Confirm box-size convergence.
9. Test multiple independent geometry seeds at fixed fractions.
10. Freeze production numerical settings only after 1-9 pass.
11. Run soma-only and neurite-only production controls and estimate control-derived $a,b$.
12. Use those control-derived exponents to predict the 144 mixed substrates before fitting mixed-data exponents.
13. Inspect residual structure, soma-radius dependence, diffusion-time dependence, and neurite-radius robustness.

A clean result is not merely two fitted constants: it is a demonstration that the physical simulation is numerically stable and that control-derived exponents predict mixed-substrate extracellular diffusivities without systematic residual structure.